In [ ]:
# Load the TOGA classification table (fix delimiter issue)
import pandas as pd

df = pd.read_csv('original_toga_classification_table.tsv', sep=',', engine='python')

In [ ]:
# Display basic info
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Filter for ORTH and PARA labels only
df_filtered = df[df['label'].isin(['ORTH', 'PARA'])]

# Show counts
print(f"Total rows after filtering: {len(df_filtered)}")
print(f"\nRows per class:")
print(df_filtered['label'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

# Create 2 subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot for single_exon = 0 (multi-exon)
for label, color in [('ORTH', 'blue'), ('PARA', 'red')]:
    data = df_filtered[(df_filtered['label'] == label) & (df_filtered['single_exon'] == 0)]
    ax1.scatter(data['gl_exo'], data['synt_log'],
                alpha=0.3, s=10, c=color, label=label)

ax1.set_xlabel('gl_exo')
ax1.set_ylabel('synt_log')
ax1.set_title('Multi-exon genes (single_exon = 0)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot for single_exon = 1
for label, color in [('ORTH', 'blue'), ('PARA', 'red')]:
    data = df_filtered[(df_filtered['label'] == label) & (df_filtered['single_exon'] == 1)]
    ax2.scatter(data['gl_exo'], data['synt_log'],
                alpha=0.3, s=10, c=color, label=label)

ax2.set_xlabel('gl_exo')
ax2.set_ylabel('synt_log')
ax2.set_title('Single-exon genes (single_exon = 1)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Split analysis by single_exon

print("=" * 80)
print("SINGLE-EXON ANALYSIS")
print("=" * 80)

# Filter suspicious PARA cases - single exon
suspicious_para_single = df_filtered[
    (df_filtered['label'] == 'PARA') &
    (df_filtered['synt_log'] > 1.0) &
    (df_filtered['gl_exo'] < 0.25) &
    (df_filtered['single_exon'] == 1)
    ]

# Get similar ORTH cases - single exon
similar_orth_single = df_filtered[
    (df_filtered['label'] == 'ORTH') &
    (df_filtered['synt_log'] > 1.0) &
    (df_filtered['gl_exo'] < 0.25) &
    (df_filtered['single_exon'] == 1)
    ]

print(f"\nSuspicious PARA cases (single-exon): {len(suspicious_para_single)}")
print(f"Similar ORTH cases (single-exon): {len(similar_orth_single)}")

# Compare features for single-exon
sample_size = 100
para_sample_single = suspicious_para_single.sample(min(sample_size, len(suspicious_para_single)), random_state=42)
orth_sample_single = similar_orth_single.sample(min(sample_size, len(similar_orth_single)), random_state=42)

feature_cols = ['gl_exo', 'exlen_to_qlen', 'synteny', 'loc_exo', 'flank_cov',
                'exon_perc', 'intr_perc', 'ex_num', 'gene_span_len', 'synt_log']

comparison_single = pd.DataFrame({
    'feature': feature_cols,
    'PARA_mean': [para_sample_single[col].mean() for col in feature_cols],
    'ORTH_mean': [orth_sample_single[col].mean() for col in feature_cols],
    'PARA_median': [para_sample_single[col].median() for col in feature_cols],
    'ORTH_median': [orth_sample_single[col].median() for col in feature_cols],
})

comparison_single['diff_mean'] = comparison_single['PARA_mean'] - comparison_single['ORTH_mean']
comparison_single['diff_median'] = comparison_single['PARA_median'] - comparison_single['ORTH_median']

print("\nSINGLE-EXON feature comparison:")
print(comparison_single.to_string(index=False))

print("\n" + "=" * 80)
print("MULTI-EXON ANALYSIS")
print("=" * 80)

# Filter suspicious PARA cases - multi exon
suspicious_para_multi = df_filtered[
    (df_filtered['label'] == 'PARA') &
    (df_filtered['synt_log'] > 1.0) &
    (df_filtered['gl_exo'] < 0.25) &
    (df_filtered['single_exon'] == 0)
    ]

# Get similar ORTH cases - multi exon
similar_orth_multi = df_filtered[
    (df_filtered['label'] == 'ORTH') &
    (df_filtered['synt_log'] > 1.0) &
    (df_filtered['gl_exo'] < 0.25) &
    (df_filtered['single_exon'] == 0)
    ]

print(f"\nSuspicious PARA cases (multi-exon): {len(suspicious_para_multi)}")
print(f"Similar ORTH cases (multi-exon): {len(similar_orth_multi)}")

# Compare features for multi-exon
para_sample_multi = suspicious_para_multi.sample(min(sample_size, len(suspicious_para_multi)), random_state=42)
orth_sample_multi = similar_orth_multi.sample(min(sample_size, len(similar_orth_multi)), random_state=42)

comparison_multi = pd.DataFrame({
    'feature': feature_cols,
    'PARA_mean': [para_sample_multi[col].mean() for col in feature_cols],
    'ORTH_mean': [orth_sample_multi[col].mean() for col in feature_cols],
    'PARA_median': [para_sample_multi[col].median() for col in feature_cols],
    'ORTH_median': [orth_sample_multi[col].median() for col in feature_cols],
})

comparison_multi['diff_mean'] = comparison_multi['PARA_mean'] - comparison_multi['ORTH_mean']
comparison_multi['diff_median'] = comparison_multi['PARA_median'] - comparison_multi['ORTH_median']

print("\nMULTI-EXON feature comparison:")
print(comparison_multi.to_string(index=False))

print("\n" + "=" * 80)
print("KEY DIFFERENCES BY EXON TYPE")
print("=" * 80)
print("\nBiggest discriminating features (by abs diff_mean):")
print("\nSingle-exon:")
print(comparison_single.nlargest(5, 'diff_mean', keep='all')[['feature', 'diff_mean', 'diff_median']])
print("\nMulti-exon:")
print(comparison_multi.nlargest(5, 'diff_mean', keep='all')[['feature', 'diff_mean', 'diff_median']])


In [ ]:
# Focus on contradictory cases, especially the two specific transcripts mentioned

print("=" * 80)
print("INVESTIGATING SPECIFIC CONTRADICTORY CASES")
print("=" * 80)

# Look for the specific transcripts
specific_transcripts = ['U_ENSG00000245532.13', 'U_ENSG00000251562.13']

specific_cases = df_filtered[df_filtered['transcript_id'].isin(specific_transcripts)]

print(f"\nFound {len(specific_cases)} cases for these transcripts:")
print(specific_cases[['chain_id', 'transcript_id', 'label', 'pred', 'synteny', 'gl_exo',
                      'flank_cov', 'loc_exo', 'single_exon', 'exon_perc', 'intr_perc']].to_string())

# Also look at all of chain 74
print("\n" + "=" * 80)
print("ALL ENTRIES FROM CHAIN 74")
print("=" * 80)

chain_74 = df_filtered[df_filtered['chain_id'] == 74]
print(f"\nTotal entries in chain 74: {len(chain_74)}")
print(f"ORTH: {len(chain_74[chain_74['label'] == 'ORTH'])}")
print(f"PARA: {len(chain_74[chain_74['label'] == 'PARA'])}")

print("\nChain 74 entries:")
print(chain_74[['transcript_id', 'label', 'pred', 'synteny', 'gl_exo', 'flank_cov',
                'loc_exo', 'single_exon', 'exon_perc', 'intr_perc', 'ex_num']].to_string())

# Now look at more contradictory cases where:
# - True label is ORTH but model predicts PARA (false negatives)
# - But features look good (high synteny, low gl_exo, good flank_cov)

print("\n" + "=" * 80)
print("FALSE NEGATIVES: True ORTH labeled as PARA by model")
print("=" * 80)

false_negatives = df_filtered[
    (df_filtered['label'] == 'ORTH') &
    (df_filtered['pred'] < 0.5)  # Model predicts PARA
    ]

print(f"\nTotal false negatives: {len(false_negatives)}")

# Among false negatives, find ones with GOOD features (should be obvious orthologs)
obvious_orthos_mislabeled = false_negatives[
    (false_negatives['synteny'] > 10) &
    (false_negatives['gl_exo'] < 0.5) &
    (false_negatives['flank_cov'] > 0.2)
    ]

print(f"False negatives with strong ortholog features: {len(obvious_orthos_mislabeled)}")

if len(obvious_orthos_mislabeled) > 0:
    print("\nTop 20 examples:")
    print(obvious_orthos_mislabeled.nlargest(20, 'synteny')[
              ['chain_id', 'transcript_id', 'label', 'pred', 'synteny', 'gl_exo',
               'flank_cov', 'loc_exo', 'single_exon', 'exon_perc', 'intr_perc']
          ].to_string())

# Compare with true PARALOGs that have similar features
print("\n" + "=" * 80)
print("TRUE PARALOGS with similar feature ranges (for comparison)")
print("=" * 80)

similar_paras = df_filtered[
    (df_filtered['label'] == 'PARA') &
    (df_filtered['synteny'] > 10) &
    (df_filtered['gl_exo'] < 0.5) &
    (df_filtered['flank_cov'] > 0.2)
    ]

print(f"True PARALOGs with these features: {len(similar_paras)}")

if len(similar_paras) > 0:
    print("\nSample of 10:")
    print(similar_paras.sample(min(10, len(similar_paras)), random_state=42)[
              ['chain_id', 'transcript_id', 'label', 'pred', 'synteny', 'gl_exo',
               'flank_cov', 'loc_exo', 'single_exon', 'exon_perc', 'intr_perc']
          ].to_string())


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np

# Test different threshold combinations
print("=" * 80)
print("TESTING DIFFERENT THRESHOLD COMBINATIONS")
print("=" * 80)

# Use full dataset for comprehensive testing
test_data = df_filtered.copy()

# Define threshold configurations to test
threshold_configs = [
    {'name': 'Original', 'synteny': 5, 'gl_exo': 0.6, 'flank_cov': 0.1},
    {'name': 'Stricter synteny', 'synteny': 10, 'gl_exo': 0.6, 'flank_cov': 0.1},
    {'name': 'Stricter gl_exo', 'synteny': 5, 'gl_exo': 0.5, 'flank_cov': 0.1},
    {'name': 'Stricter flank', 'synteny': 5, 'gl_exo': 0.6, 'flank_cov': 0.2},
    {'name': 'Relaxed gl_exo', 'synteny': 5, 'gl_exo': 0.7, 'flank_cov': 0.1},
    {'name': 'Combined strict', 'synteny': 10, 'gl_exo': 0.5, 'flank_cov': 0.2},
    {'name': 'Moderate', 'synteny': 7, 'gl_exo': 0.55, 'flank_cov': 0.15},
]

results = []

for config in threshold_configs:
    # Apply rule: call ORTH if ALL conditions met
    def classifier(row):
        if (row['synteny'] > config['synteny'] and
                row['gl_exo'] < config['gl_exo'] and
                row['flank_cov'] > config['flank_cov']):
            return 'ORTH'
        else:
            return 'PARA'


    test_data['rule_pred'] = test_data.apply(classifier, axis=1)

    # Calculate metrics
    cm = confusion_matrix(test_data['label'], test_data['rule_pred'], labels=['ORTH', 'PARA'])
    accuracy = accuracy_score(test_data['label'], test_data['rule_pred'])

    # Calculate precision, recall, F1 for ORTH
    tp = cm[0, 0]  # True ORTH predicted as ORTH
    fp = cm[1, 0]  # True PARA predicted as ORTH
    fn = cm[0, 1]  # True ORTH predicted as PARA
    tn = cm[1, 1]  # True PARA predicted as PARA

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    results.append({
        'config': config['name'],
        'synteny_th': config['synteny'],
        'gl_exo_th': config['gl_exo'],
        'flank_th': config['flank_cov'],
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn
    })

# Display results as a table
results_df = pd.DataFrame(results)
print("\nPERFORMANCE COMPARISON:")
print(results_df.to_string(index=False))

print("\n" + "=" * 80)
print("BEST CONFIGURATIONS")
print("=" * 80)
print("\nBest by F1 score:")
print(results_df.nlargest(3, 'f1')[['config', 'synteny_th', 'gl_exo_th', 'flank_th', 'f1', 'precision', 'recall']])

print("\nBest by accuracy:")
print(results_df.nlargest(3, 'accuracy')[
          ['config', 'synteny_th', 'gl_exo_th', 'flank_th', 'accuracy', 'precision', 'recall']])

# Check the specific problematic cases
print("\n" + "=" * 80)
print("CHECKING PROBLEMATIC CASES (Chain 74, transcripts 245532 & 251562)")
print("=" * 80)

specific_cases = df_filtered[df_filtered['transcript_id'].isin(['U_ENSG00000245532.13', 'U_ENSG00000251562.13'])]

for config in threshold_configs:
    def classifier(row):
        if (row['synteny'] > config['synteny'] and
                row['gl_exo'] < config['gl_exo'] and
                row['flank_cov'] > config['flank_cov']):
            return 'ORTH'
        else:
            return 'PARA'


    specific_cases[f"pred_{config['name']}"] = specific_cases.apply(classifier, axis=1)

print("\nPredictions for problematic cases:")
pred_cols = [col for col in specific_cases.columns if col.startswith('pred_')]
print(specific_cases[['transcript_id', 'label', 'synteny', 'gl_exo', 'flank_cov'] + pred_cols])


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

print("=" * 80)
print("TRAINING LOGISTIC REGRESSION MODEL FOR lncRNA CLASSIFICATION")
print("=" * 80)

# Prepare data
# Add log-transformed synteny feature
df_filtered['synteny_log1p'] = np.log1p(df_filtered['synteny'])

# Select features
feature_names = ['synteny_log1p', 'gl_exo', 'flank_cov']
X = df_filtered[feature_names].values
y = (df_filtered['label'] == 'ORTH').astype(int)  # 1 for ORTH, 0 for PARA

# Add hard constraint: gl_exo > 0.95 is always PARA
hard_para_mask = df_filtered['gl_exo'] > 0.95
print(f"\nHard constraint: {hard_para_mask.sum()} cases with gl_exo > 0.95 will be forced to PARA")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train logistic regression
# Use class_weight='balanced' to handle class imbalance
logreg = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
logreg.fit(X_train, y_train)

# Get coefficients
coeffs = logreg.coef_[0]
intercept = logreg.intercept_[0]

print("\n" + "=" * 80)
print("MODEL COEFFICIENTS (scoring function)")
print("=" * 80)
print(
    f"\nScore = {coeffs[0]:.4f} * log1p(synteny) + {coeffs[1]:.4f} * gl_exo + {coeffs[2]:.4f} * flank_cov + {intercept:.4f}")
print("\nInterpretation:")
print(
    f"  synteny_log1p: {coeffs[0]:.4f} ({'positive' if coeffs[0] > 0 else 'negative'}) - {'higher synteny → ORTH' if coeffs[0] > 0 else 'higher synteny → PARA'}")
print(
    f"  gl_exo:        {coeffs[1]:.4f} ({'positive' if coeffs[1] > 0 else 'negative'}) - {'higher gl_exo → ORTH' if coeffs[1] > 0 else 'higher gl_exo → PARA'}")
print(
    f"  flank_cov:     {coeffs[2]:.4f} ({'positive' if coeffs[2] > 0 else 'negative'}) - {'higher flank_cov → ORTH' if coeffs[2] > 0 else 'higher flank_cov → PARA'}")

# Predict on test set
y_pred = logreg.predict(X_test)
y_pred_proba = logreg.predict_proba(X_test)[:, 1]  # Probability of ORTH

# Apply hard constraint
test_indices = df_filtered.iloc[X_test.shape[0]:].index
hard_para_test = df_filtered.loc[test_indices[hard_para_mask.iloc[X_test.shape[0]:].values], 'gl_exo'] > 0.95
if hard_para_test.sum() > 0:
    y_pred[hard_para_test] = 0

print("\n" + "=" * 80)
print("MODEL PERFORMANCE ON TEST SET")
print("=" * 80)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(f"               Predicted PARA  Predicted ORTH")
print(f"True PARA      {cm[0, 0]:14d}  {cm[0, 1]:14d}")
print(f"True ORTH      {cm[1, 0]:14d}  {cm[1, 1]:14d}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['PARA', 'ORTH']))

# ROC AUC
auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nROC AUC Score: {auc:.4f}")

# Compare with original model on test set
test_df = df_filtered.iloc[-len(y_test):]
orig_pred = (test_df['pred'] > 0.5).astype(int)
print("\n" + "=" * 80)
print("COMPARISON WITH ORIGINAL MODEL")
print("=" * 80)
print("\nOriginal Model:")
cm_orig = confusion_matrix(y_test, orig_pred)
print(f"               Predicted PARA  Predicted ORTH")
print(f"True PARA      {cm_orig[0, 0]:14d}  {cm_orig[0, 1]:14d}")
print(f"True ORTH      {cm_orig[1, 0]:14d}  {cm_orig[1, 1]:14d}")

# Test on the specific problematic cases
print("\n" + "=" * 80)
print("TESTING ON PROBLEMATIC CASES (Chain 74)")
print("=" * 80)

specific = df_filtered[df_filtered['transcript_id'].isin(['U_ENSG00000245532.13', 'U_ENSG00000251562.13'])]
X_specific = specific[feature_names].values
specific_pred = logreg.predict(X_specific)
specific_proba = logreg.predict_proba(X_specific)[:, 1]

for i, (idx, row) in enumerate(specific.iterrows()):
    print(f"\n{row['transcript_id']}:")
    print(f"  True label: {row['label']}")
    print(f"  Original model: {'ORTH' if row['pred'] > 0.5 else 'PARA'} (prob={row['pred']:.4f})")
    print(f"  New LogReg: {'ORTH' if specific_pred[i] == 1 else 'PARA'} (prob={specific_proba[i]:.4f})")
    print(f"  Features: synteny={row['synteny']}, gl_exo={row['gl_exo']:.3f}, flank_cov={row['flank_cov']:.3f}")
    score = coeffs[0] * np.log1p(row['synteny']) + coeffs[1] * row['gl_exo'] + coeffs[2] * row['flank_cov'] + intercept
    print(f"  Raw score: {score:.4f}")

# Plot ROC curve
print("\n" + "=" * 80)
print("PLOTTING ROC CURVE")
print("=" * 80)

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, label=f'LogReg (AUC = {auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve: Logistic Regression for lncRNA Ortholog Classification')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nModel training complete!")


In [ ]:
print("=" * 80)
print("CREATING COMPARISON DATASET FOR MANUAL VALIDATION")
print("=" * 80)

# Sample diverse ORTH cases (original model predicts as ORTH)
orth_original = df_filtered[df_filtered['label'] == 'ORTH'].copy()

# Stratified sampling to get diverse cases across feature space
# Bin by gl_exo, synteny, and flank_cov to ensure diversity
orth_original['gl_exo_bin'] = pd.cut(orth_original['gl_exo'], bins=[0, 0.2, 0.4, 0.6, 1.0])
orth_original['synteny_bin'] = pd.cut(orth_original['synteny'], bins=[0, 10, 30, 100, 1000])
orth_original['flank_bin'] = pd.cut(orth_original['flank_cov'], bins=[0, 0.2, 0.5, 1.0])

# Sample 1000 diverse ORTH cases
orth_sample_diverse = orth_original.groupby(['gl_exo_bin', 'synteny_bin', 'flank_bin'],
                                            observed=True).apply(
    lambda x: x.sample(min(len(x), 50), random_state=42)
).sample(min(1000, len(orth_original)), random_state=42)

print(f"\nSampled {len(orth_sample_diverse)} diverse ORTH cases")

# Sample diverse PARA cases (original model predicts as PARA)
para_original = df_filtered[df_filtered['label'] == 'PARA'].copy()

para_original['gl_exo_bin'] = pd.cut(para_original['gl_exo'], bins=[0, 0.2, 0.4, 0.6, 1.0])
para_original['synteny_bin'] = pd.cut(para_original['synteny'], bins=[0, 10, 30, 100, 1000])
para_original['flank_bin'] = pd.cut(para_original['flank_cov'], bins=[0, 0.2, 0.5, 1.0])

# Sample 1000 diverse PARA cases
para_sample_diverse = para_original.groupby(['gl_exo_bin', 'synteny_bin', 'flank_bin'],
                                            observed=True).apply(
    lambda x: x.sample(min(len(x), 50), random_state=42)
).sample(min(998, len(para_original)), random_state=42)  # 998 to leave room for 2 specific cases

print(f"Sampled {len(para_sample_diverse)} diverse PARA cases")

# Add the 2 problematic cases to PARA sample
problematic = df_filtered[df_filtered['transcript_id'].isin(['U_ENSG00000245532.13',
                                                             'U_ENSG00000251562.13'])]
para_sample_diverse = pd.concat([para_sample_diverse, problematic])

print(f"Added 2 problematic cases, now {len(para_sample_diverse)} PARA cases total")

# Combine into comparison dataset
comparison_set = pd.concat([orth_sample_diverse, para_sample_diverse])

# Select only needed columns
comparison_df = comparison_set[['chain_id', 'transcript_id', 'gl_exo', 'synteny',
                                'synt_log', 'flank_cov', 'label', 'pred']].copy()

# Add original model prediction label
comparison_df['original_pred'] = np.where(comparison_df['pred'] > 0.5, 'ORTH', 'PARA')

# Add new LogReg prediction
X_comp = comparison_df[['synteny', 'gl_exo', 'flank_cov']].copy()
X_comp['synteny_log1p'] = np.log1p(X_comp['synteny'])
X_comp_features = X_comp[['synteny_log1p', 'gl_exo', 'flank_cov']].values

logreg_pred = logreg.predict(X_comp_features)
logreg_proba = logreg.predict_proba(X_comp_features)[:, 1]

comparison_df['logreg_pred'] = np.where(logreg_pred == 1, 'ORTH', 'PARA')
comparison_df['logreg_prob'] = logreg_proba

# Identify disagreements
comparison_df['disagree'] = comparison_df['original_pred'] != comparison_df['logreg_pred']

# Sort by disagreements first, then by chain_id
comparison_df = comparison_df.sort_values(['disagree', 'chain_id', 'transcript_id'],
                                          ascending=[False, True, True])

print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
print(f"\nTotal cases: {len(comparison_df)}")
print(f"  ORTH (original label): {(comparison_df['label'] == 'ORTH').sum()}")
print(f"  PARA (original label): {(comparison_df['label'] == 'PARA').sum()}")

print(f"\nDisagreements: {comparison_df['disagree'].sum()} " +
      f"({100 * comparison_df['disagree'].sum() / len(comparison_df):.1f}%)")

print("\nDisagreement breakdown:")
disagree_summary = comparison_df[comparison_df['disagree']].groupby(
    ['label', 'original_pred', 'logreg_pred']
).size().reset_index(name='count')
print(disagree_summary.to_string(index=False))

# Show first 30 disagreements
print("\n" + "=" * 80)
print("FIRST 30 DISAGREEMENTS FOR MANUAL REVIEW")
print("=" * 80)

disagreements_df = comparison_df[comparison_df['disagree']].head(30)
print(disagreements_df[['chain_id', 'transcript_id', 'label', 'synteny', 'gl_exo',
                        'flank_cov', 'original_pred', 'logreg_pred', 'logreg_prob']].to_string(index=False))

# Save to file for manual review
output_file = '/Users/Bogdan.Kirilenko/PycharmProjects/CURIA/toga_model_tune/comparison_for_review.tsv'
comparison_df.to_csv(output_file, sep='\t', index=False)

print(f"\n" + "=" * 80)
print(f"Full comparison saved to: {output_file}")
print("=" * 80)
print("\nColumns in output file:")
print("  chain_id, transcript_id, gl_exo, synteny, synt_log, flank_cov")
print("  label (original), pred (original probability)")
print("  original_pred (ORTH/PARA), logreg_pred (ORTH/PARA), logreg_prob")
print("  disagree (True/False)")

print("\n" + "=" * 80)
print("NOTE: The 2 problematic cases from Chain 74:")
print("=" * 80)
prob_in_comparison = comparison_df[
    comparison_df['transcript_id'].isin(['U_ENSG00000245532.13', 'U_ENSG00000251562.13'])]
print(prob_in_comparison[['chain_id', 'transcript_id', 'label', 'synteny', 'gl_exo',
                          'flank_cov', 'original_pred', 'logreg_pred', 'logreg_prob']].to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt

# Get all disagreements
all_disagreements = comparison_df[comparison_df['disagree']].copy()

# Create disagreement type column
all_disagreements['disagree_type'] = all_disagreements.apply(
    lambda row: f"{row['original_pred']}→{row['logreg_pred']}", axis=1
)

print("=" * 80)
print("PLOTTING DISAGREEMENTS: gl_exo vs synteny")
print("=" * 80)
print(f"\nTotal disagreements: {len(all_disagreements)}")
print(f"  ORTH→PARA: {len(all_disagreements[all_disagreements['disagree_type'] == 'ORTH→PARA'])}")
print(f"  PARA→ORTH: {len(all_disagreements[all_disagreements['disagree_type'] == 'PARA→ORTH'])}")

# Create plot
fig, ax = plt.subplots(figsize=(12, 8))

# Plot ORTH→PARA (original said ORTH, new says PARA)
orth_to_para = all_disagreements[all_disagreements['disagree_type'] == 'ORTH→PARA']
ax.scatter(orth_to_para['gl_exo'], orth_to_para['synteny'],
           alpha=0.6, s=80, c='red', marker='v',
           label=f'ORTH→PARA (n={len(orth_to_para)})', edgecolors='darkred', linewidths=1)

# Plot PARA→ORTH (original said PARA, new says ORTH)
para_to_orth = all_disagreements[all_disagreements['disagree_type'] == 'PARA→ORTH']
ax.scatter(para_to_orth['gl_exo'], para_to_orth['synteny'],
           alpha=0.6, s=80, c='blue', marker='^',
           label=f'PARA→ORTH (n={len(para_to_orth)})', edgecolors='darkblue', linewidths=1)

# Highlight the 2 problematic cases if they're in disagreements
problematic_in_disagree = all_disagreements[
    all_disagreements['transcript_id'].isin(['U_ENSG00000245532.13', 'U_ENSG00000251562.13'])
]

if len(problematic_in_disagree) > 0:
    ax.scatter(problematic_in_disagree['gl_exo'], problematic_in_disagree['synteny'],
               s=300, c='gold', marker='*',
               label='Problematic Chain 74 cases', edgecolors='black', linewidths=2)

ax.set_xlabel('gl_exo', fontsize=12)
ax.set_ylabel('synteny', fontsize=12)
ax.set_title('Disagreements between Original Model and LogReg\n(gl_exo vs synteny)', fontsize=14)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

# Add reference lines
ax.axhline(y=10, color='gray', linestyle='--', alpha=0.5, label='synteny=10')
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='gl_exo=0.5')

plt.tight_layout()
plt.show()

# Print statistics by disagreement type
print("\n" + "=" * 80)
print("STATISTICS BY DISAGREEMENT TYPE")
print("=" * 80)

for disagree_type in ['ORTH→PARA', 'PARA→ORTH']:
    subset = all_disagreements[all_disagreements['disagree_type'] == disagree_type]
    if len(subset) > 0:
        print(f"\n{disagree_type} ({len(subset)} cases):")
        print(f"  synteny:   mean={subset['synteny'].mean():.1f}, median={subset['synteny'].median():.1f}, " +
              f"range=[{subset['synteny'].min():.0f}, {subset['synteny'].max():.0f}]")
        print(f"  gl_exo:    mean={subset['gl_exo'].mean():.3f}, median={subset['gl_exo'].median():.3f}, " +
              f"range=[{subset['gl_exo'].min():.3f}, {subset['gl_exo'].max():.3f}]")
        print(f"  flank_cov: mean={subset['flank_cov'].mean():.3f}, median={subset['flank_cov'].median():.3f}, " +
              f"range=[{subset['flank_cov'].min():.3f}, {subset['flank_cov'].max():.3f}]")


# Subsample lncRNA training set

In [24]:
print("=" * 80)
print("FILTERING FOR lncRNA TRANSCRIPTS ONLY")
print("=" * 80)

# Load biotype information
biotypes = pd.read_csv('isoforms_union.tsv', sep='\t')
print(f"\nLoaded biotype information for {len(biotypes)} transcripts")
print(f"\nBiotype distribution:")
print(biotypes['biotype'].value_counts().head(10))

# Get set of lncRNA transcript IDs
lncrna_transcripts = set(biotypes[biotypes['biotype'] == 'lncRNA']['transcript_id'])
print(f"\nFound {len(lncrna_transcripts)} lncRNA transcripts")

# Filter df_filtered to only lncRNAs
df_lncrna = df_filtered[df_filtered['transcript_id'].isin(lncrna_transcripts)].copy()

print(f"\nFiltered data:")
print(f"  Total lncRNA entries: {len(df_lncrna)}")
print(f"  ORTH: {(df_lncrna['label'] == 'ORTH').sum()}")
print(f"  PARA: {(df_lncrna['label'] == 'PARA').sum()}")

# Now sample 1000 ORTH and 1000 PARA from lncRNAs only
print("\n" + "=" * 80)
print("SAMPLING DIVERSE lncRNA CASES")
print("=" * 80)

# Sample ORTH - simple random sample for diversity
orth_lncrna = df_lncrna[df_lncrna['label'] == 'ORTH']
orth_sample_lncrna = orth_lncrna.sample(min(1000, len(orth_lncrna)), random_state=42)
print(f"Sampled {len(orth_sample_lncrna)} lncRNA ORTH cases")

# Sample PARA
para_lncrna = df_lncrna[df_lncrna['label'] == 'PARA']

# Check if problematic cases exist and reserve space for them
problematic_lncrna = df_lncrna[df_lncrna['transcript_id'].isin(['U_ENSG00000245532.13',
                                                                'U_ENSG00000251562.13'])]
num_problematic = len(problematic_lncrna)
num_para_to_sample = min(1000 - num_problematic, len(para_lncrna))

para_sample_lncrna = para_lncrna.sample(num_para_to_sample, random_state=42)
print(f"Sampled {len(para_sample_lncrna)} lncRNA PARA cases")

if num_problematic > 0:
    para_sample_lncrna = pd.concat([para_sample_lncrna, problematic_lncrna])
    print(f"Added {num_problematic} problematic cases from Chain 74")

# Combine
comparison_set_lncrna = pd.concat([orth_sample_lncrna, para_sample_lncrna])

# Create comparison dataframe
comparison_df_lncrna = comparison_set_lncrna[['chain_id', 'transcript_id', 'gl_exo', 'synteny',
                                              'synt_log', 'flank_cov', 'label', 'pred']].copy()

comparison_df_lncrna['original_pred'] = np.where(comparison_df_lncrna['pred'] > 0.5, 'ORTH', 'PARA')

# Add LogReg predictions
X_comp_lncrna = comparison_df_lncrna[['synteny', 'gl_exo', 'flank_cov']].copy()
X_comp_lncrna['synteny_log1p'] = np.log1p(X_comp_lncrna['synteny'])
X_comp_features_lncrna = X_comp_lncrna[['synteny_log1p', 'gl_exo', 'flank_cov']].values

logreg_pred_lncrna = logreg.predict(X_comp_features_lncrna)
logreg_proba_lncrna = logreg.predict_proba(X_comp_features_lncrna)[:, 1]

comparison_df_lncrna['logreg_pred'] = np.where(logreg_pred_lncrna == 1, 'ORTH', 'PARA')
comparison_df_lncrna['logreg_prob'] = logreg_proba_lncrna
comparison_df_lncrna['disagree'] = comparison_df_lncrna['original_pred'] != comparison_df_lncrna['logreg_pred']

# Sort by disagreements
comparison_df_lncrna = comparison_df_lncrna.sort_values(['disagree', 'chain_id', 'transcript_id'],
                                                        ascending=[False, True, True])

print("\n" + "=" * 80)
print("lncRNA COMPARISON SUMMARY")
print("=" * 80)
print(f"\nTotal cases: {len(comparison_df_lncrna)}")
print(f"  ORTH: {(comparison_df_lncrna['label'] == 'ORTH').sum()}")
print(f"  PARA: {(comparison_df_lncrna['label'] == 'PARA').sum()}")
print(f"\nDisagreements: {comparison_df_lncrna['disagree'].sum()} " +
      f"({100 * comparison_df_lncrna['disagree'].sum() / len(comparison_df_lncrna):.1f}%)")

disagree_summary_lncrna = comparison_df_lncrna[comparison_df_lncrna['disagree']].groupby(
    ['label', 'original_pred', 'logreg_pred']
).size().reset_index(name='count')
print("\nDisagreement breakdown:")
print(disagree_summary_lncrna.to_string(index=False))

# Save to file
output_file_lncrna = '/Users/Bogdan.Kirilenko/PycharmProjects/CURIA/toga_model_tune/lncrna_comparison_for_review.tsv'
comparison_df_lncrna.to_csv(output_file_lncrna, sep='\t', index=False)
print(f"\nlncRNA comparison saved to: {output_file_lncrna}")


FILTERING FOR lncRNA TRANSCRIPTS ONLY

Loaded biotype information for 79317 transcripts

Biotype distribution:
biotype
lncRNA                                34780
protein_coding                        19746
processed_pseudogene                   9486
misc_RNA                               2207
unprocessed_pseudogene                 1953
snRNA                                  1901
miRNA                                  1879
transcribed_unprocessed_pseudogene     1596
transcribed_processed_pseudogene       1154
TEC                                    1024
Name: count, dtype: int64

Found 34780 lncRNA transcripts

Filtered data:
  Total lncRNA entries: 77813
  ORTH: 37433
  PARA: 40380

SAMPLING DIVERSE lncRNA CASES
Sampled 1000 lncRNA ORTH cases
Sampled 998 lncRNA PARA cases
Added 2 problematic cases from Chain 74

lncRNA COMPARISON SUMMARY

Total cases: 2000
  ORTH: 1000
  PARA: 1000

Disagreements: 161 (8.1%)

Disagreement breakdown:
label original_pred logreg_pred  count
 ORTH         

In [26]:
import plotly.graph_objects as go

print("=" * 80)
print("CREATING 3D PLOTLY VISUALIZATION")
print("=" * 80)

# Create disagree_type column
comparison_df_lncrna['disagree_type'] = 'Agreement'
comparison_df_lncrna.loc[
    (comparison_df_lncrna['disagree']) & (comparison_df_lncrna['original_pred'] == 'ORTH'),
    'disagree_type'] = 'ORTH→PARA'
comparison_df_lncrna.loc[
    (comparison_df_lncrna['disagree']) & (comparison_df_lncrna['original_pred'] == 'PARA'),
    'disagree_type'] = 'PARA→ORTH'

print(f"\nData breakdown:")
print(f"  Agreement: {(comparison_df_lncrna['disagree_type'] == 'Agreement').sum()}")
print(f"  ORTH→PARA: {(comparison_df_lncrna['disagree_type'] == 'ORTH→PARA').sum()}")
print(f"  PARA→ORTH: {(comparison_df_lncrna['disagree_type'] == 'PARA→ORTH').sum()}")

# Create 3D scatter plot
fig = go.Figure()

# Plot each disagree type separately
for disagree_type, color, opacity in [('Agreement', 'lightgray', 0.3),
                                       ('ORTH→PARA', 'red', 0.8),
                                       ('PARA→ORTH', 'blue', 0.8)]:
    subset = comparison_df_lncrna[comparison_df_lncrna['disagree_type'] == disagree_type]
    
    # Create hover text
    hover_text = [
        f"Transcript: {tid}<br>" +
        f"Chain: {cid}<br>" +
        f"gl_exo: {glex:.3f}<br>" +
        f"synteny: {synt}<br>" +
        f"flank_cov: {flank:.3f}<br>" +
        f"Original: {opred}<br>" +
        f"LogReg: {lpred} ({lprob:.3f})"
        for tid, cid, glex, synt, flank, opred, lpred, lprob in zip(
            subset['transcript_id'],
            subset['chain_id'],
            subset['gl_exo'],
            subset['synteny'],
            subset['flank_cov'],
            subset['original_pred'],
            subset['logreg_pred'],
            subset['logreg_prob']
        )
    ]
    
    fig.add_trace(go.Scatter3d(
        x=subset['gl_exo'],
        y=subset['synteny'],
        z=subset['flank_cov'],
        mode='markers',
        name=f'{disagree_type} (n={len(subset)})',
        marker=dict(
            size=5 if disagree_type == 'Agreement' else 7,
            color=color,
            opacity=opacity,
            line=dict(width=0 if disagree_type == 'Agreement' else 0.5, color='black')
        ),
        text=hover_text,
        hovertemplate='%{text}<extra></extra>'
    ))

# Update layout
fig.update_layout(
    title='lncRNA Model Disagreements: 3D Feature Space<br>' +
          '<sub>gl_exo vs synteny vs flank_cov</sub>',
    scene=dict(
        xaxis_title='gl_exo',
        yaxis_title='synteny',
        zaxis_title='flank_cov',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.3)
        )
    ),
    width=1000,
    height=800,
    showlegend=True,
    legend=dict(
        x=0.7,
        y=0.95,
        bgcolor='rgba(255, 255, 255, 0.8)'
    )
)

fig.show()

print("\n3D visualization created! Rotate to explore the feature space.")

CREATING 3D PLOTLY VISUALIZATION

Data breakdown:
  Agreement: 1839
  ORTH→PARA: 31
  PARA→ORTH: 130



3D visualization created! Rotate to explore the feature space.


In [28]:
# View all disagreement cases for lncRNA
disagreements_lncrna = comparison_df_lncrna[comparison_df_lncrna['disagree']].copy()

print("=" * 80)
print(f"ALL {len(disagreements_lncrna)} DISAGREEMENT CASES")
print("=" * 80)

# Split by disagreement type
orth_to_para_lncrna = disagreements_lncrna[disagreements_lncrna['disagree_type'] == 'ORTH→PARA']
para_to_orth_lncrna = disagreements_lncrna[disagreements_lncrna['disagree_type'] == 'PARA→ORTH']

print(f"\n{'=' * 80}")
print(f"ORTH→PARA cases ({len(orth_to_para_lncrna)})")
print(f"Original model says ORTH, LogReg says PARA")
print(f"{'=' * 80}\n")
print(orth_to_para_lncrna[['chain_id', 'transcript_id', 'synteny', 'gl_exo',
                           'flank_cov', 'pred', 'logreg_prob']].sort_values('synteny').to_string(index=False))

print(f"\n\n{'=' * 80}")
print(f"PARA→ORTH cases ({len(para_to_orth_lncrna)})")
print(f"Original model says PARA, LogReg says ORTH")
print(f"{'=' * 80}\n")
print(para_to_orth_lncrna[['chain_id', 'transcript_id', 'synteny', 'gl_exo',
                           'flank_cov', 'pred', 'logreg_prob']].sort_values('synteny', ascending=False).to_string(
    index=False))

# Summary statistics by disagreement type
print(f"\n\n{'=' * 80}")
print("FEATURE STATISTICS BY DISAGREEMENT TYPE")
print(f"{'=' * 80}\n")

for dtype in ['ORTH→PARA', 'PARA→ORTH']:
    subset = disagreements_lncrna[disagreements_lncrna['disagree_type'] == dtype]
    if len(subset) > 0:
        print(f"{dtype} (n={len(subset)}):")
        print(f"  synteny:   min={subset['synteny'].min():.0f}, "
              f"median={subset['synteny'].median():.0f}, "
              f"mean={subset['synteny'].mean():.1f}, "
              f"max={subset['synteny'].max():.0f}")
        print(f"  gl_exo:    min={subset['gl_exo'].min():.3f}, "
              f"median={subset['gl_exo'].median():.3f}, "
              f"mean={subset['gl_exo'].mean():.3f}, "
              f"max={subset['gl_exo'].max():.3f}")
        print(f"  flank_cov: min={subset['flank_cov'].min():.3f}, "
              f"median={subset['flank_cov'].median():.3f}, "
              f"mean={subset['flank_cov'].mean():.3f}, "
              f"max={subset['flank_cov'].max():.3f}")
        print()


ALL 161 DISAGREEMENT CASES

ORTH→PARA cases (31)
Original model says ORTH, LogReg says PARA

 chain_id       transcript_id  synteny   gl_exo  flank_cov     pred  logreg_prob
   183189 U_ENSG00000293395.1        1 0.320731    0.04490 0.853109     0.351986
   176290 U_ENSG00000261467.3        1 0.196622    0.01350 0.543786     0.489503
   118338 U_ENSG00000234052.3        1 0.200802    0.01780 0.858148     0.499023
   375792 U_ENSG00000297960.1        2 0.289768    0.03635 0.965400     0.472851
   264602 U_ENSG00000290900.2        2 0.225464    0.00000 0.609802     0.457944
    14333 U_ENSG00000235481.5        2 0.279931    0.00000 0.575550     0.340462
    20961 U_ENSG00000235481.5        2 0.305692    0.00000 0.512760     0.290237
    31366 U_ENSG00000267745.1        2 0.458767    0.06020 0.564721     0.228593
    32975 U_ENSG00000285483.2        2 0.284379    0.00000 0.539414     0.331487
    81978 U_ENSG00000299974.1        2 0.279783    0.01200 0.646118     0.389806
   304946 U_ENSG